# LIBRARY

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sktime.transformations.series.adapt import TabularToSeriesAdaptor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.metrics import roc_curve, auc
from sktime.classification.shapelet_based import ShapeletTransformClassifier
from sktime.transformations.panel.shapelet_transform import RandomShapeletTransform
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sktime.classification.distance_based import KNeighborsTimeSeriesClassifier
from scikitplot.metrics import plot_roc
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score, classification_report
from sklearn.model_selection import GroupShuffleSplit
import numpy as np
import pandas as pd
import gzip 
import pickle 
from collections import Counter
import matplotlib.pyplot as plt
import gzip
import pickle
import sys
import numpy as np
import warnings
import pandas as pd
from scipy.stats import levene
from statsmodels.tsa.stattools import adfuller, kpss
from sklearn.preprocessing import RobustScaler
from statsmodels.tsa.seasonal import STL
from statsmodels.tools.sm_exceptions import InterpolationWarning
warnings.simplefilter("ignore", InterpolationWarning)
warnings.simplefilter("ignore", UserWarning)


# DATAFRAMES

In [2]:
with gzip.open("../1.DATASET/CMI_timeseries_personalized.pkl.gz", "rb") as f:
	 CMI_timeseries_personalized = pickle.load(f)

In [3]:
# Check how many subjects/time series
print(f"Number of time series: {len(CMI_timeseries_personalized)}")

Number of time series: 4307


## Cleaning

In [4]:
df = CMI_timeseries_personalized[0].copy()
df.columns

Index(['X', 'Y', 'Z', 'enmo', 'anglez', 'non-wear_flag', 'light',
       'battery_voltage', 'weekday', 'quarter', 'relative_date_PCIAT', 'id',
       'sii_binary'],
      dtype='object')

In [5]:
cols_to_drop = ["battery_voltage", "timestamp", "quarter", "relative_date_PCIAT"]
data_clean   = [df.drop(columns=cols_to_drop, errors="ignore") for df in CMI_timeseries_personalized]

print(f"Columns remaining: {data_clean[0].columns.tolist()}")
print(f"Total subjects   : {len(data_clean)}")

Columns remaining: ['X', 'Y', 'Z', 'enmo', 'anglez', 'non-wear_flag', 'light', 'weekday', 'id', 'sii_binary']
Total subjects   : 4307


## id check

In [6]:
all_labels = [df["sii_binary"].iloc[0] for df in data_clean]
all_ids    = [df["id"].iloc[0]         for df in data_clean]

# Filter missing labels
valid_idx    = [i for i, label in enumerate(all_labels) if not pd.isna(label)]
valid_labels = np.array([all_labels[i] for i in valid_idx])
valid_ids    = np.array([all_ids[i]    for i in valid_idx])

print(f"Total datasets          : {len(data_clean)}")
print(f"Valid (non-null label)  : {len(valid_idx)}")
print(f"Unique subject IDs      : {len(set(valid_ids))}")

Total datasets          : 4307
Valid (non-null label)  : 4307
Unique subject IDs      : 372


## split 

In [7]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_pos, test_pos = next(gss.split(
    X=valid_idx,
    y=valid_labels,
    groups=valid_ids
))

train_idx = [valid_idx[i] for i in train_pos]
test_idx  = [valid_idx[i] for i in test_pos]

train_dfs = [data_clean[i] for i in train_idx]
test_dfs  = [data_clean[i] for i in test_idx]

In [8]:
# ── Sanity checks ─────────────────────────────────────────────────────────────
train_ids = set(df["id"].iloc[0] for df in train_dfs)
test_ids  = set(df["id"].iloc[0] for df in test_dfs)
overlap   = train_ids & test_ids

print(f"\nTrain datasets  : {len(train_dfs)}")
print(f"Test datasets   : {len(test_dfs)}")
print(f"Total           : {len(train_dfs) + len(test_dfs)}")
print(f"Subject ID overlap (must be 0): {len(overlap)}")


Train datasets  : 3298
Test datasets   : 1009
Total           : 4307
Subject ID overlap (must be 0): 0


# NORMALIZATION

In [9]:
signals = ["X", "Y", "Z", "enmo", "anglez", "light", "non-wear_flag", "weekday"]

In [10]:
def fit_global_scalers(df_list, signals):
    """Fit one RobustScaler per signal using TRAIN data only."""
    global_scalers = {}
    print("Fitting global scalers on train data...")
    for signal in signals:
        all_values = []
        for df in df_list:
            if signal not in df.columns:
                continue
            y = df[signal].values
            if np.std(y) < 0.0001 or np.ptp(y) == 0:
                continue
            all_values.append(y)
        if not all_values:
            continue
        population = np.concatenate(all_values).reshape(-1, 1)
        q25, q75   = np.percentile(population, [25, 75])
        iqr        = q75 - q25
        signal_std = np.std(population)
        if iqr > 1e-4:
            scaler = RobustScaler()
            scaler.fit(population)
            global_scalers[signal] = ("robust", scaler)
            print(f"  {signal:<18} → RobustScaler   (IQR={iqr:.4f})")
        elif signal_std > 1e-6:
            global_scalers[signal] = ("std_fallback", np.median(population), signal_std)
            print(f"  {signal:<18} → Std fallback   (IQR too small)")
        else:
            global_scalers[signal] = ("zero", None)
            print(f"  {signal:<18} → Zeroed")
    return global_scalers


In [11]:
def apply_global_scalers(df_list, global_scalers):
    """Apply pre-fitted scalers to a list of DataFrames."""
    scaled_list = []
    for df in df_list:
        target_df = df.copy()
        for signal, scaler_info in global_scalers.items():
            if signal not in target_df.columns:
                continue
            y    = target_df[signal].values.copy()
            kind = scaler_info[0]
            if kind == "robust":
                _, scaler = scaler_info
                y_scaled  = scaler.transform(y.reshape(-1, 1)).flatten()
            elif kind == "std_fallback":
                _, median, std = scaler_info
                y_scaled  = (y - median) / std
            else:
                y_scaled  = np.zeros_like(y)
            target_df[signal] = y_scaled
        scaled_list.append(target_df)
    return scaled_list


In [12]:

global_scalers = fit_global_scalers(train_dfs, signals)
train_scaled   = apply_global_scalers(train_dfs, global_scalers)
test_scaled    = apply_global_scalers(test_dfs,  global_scalers)

print(f"\nTrain scaled: {len(train_scaled)} | Test scaled: {len(test_scaled)}")

Fitting global scalers on train data...
  X                  → RobustScaler   (IQR=0.8217)
  Y                  → RobustScaler   (IQR=0.4166)
  Z                  → RobustScaler   (IQR=0.4350)
  enmo               → RobustScaler   (IQR=0.0463)
  anglez             → RobustScaler   (IQR=5.6673)
  light              → RobustScaler   (IQR=1.8472)
  non-wear_flag      → RobustScaler   (IQR=1.0000)

Train scaled: 3298 | Test scaled: 1009


## split

In [13]:
FEATURE_SIGNALS = ["X", "Y", "Z", "enmo", "anglez", "non-wear_flag", "light", "weekday"]
TARGET_SIGNAL   = "sii_binary"

def build_Xy(df_list, feature_signals, target_signal):
    X_list, y_list = [], []
    for df in df_list:
        target_value = df[target_signal].iloc[0]
        if pd.isna(target_value):
            continue
        row = {s: pd.Series(df[s].values) for s in feature_signals if s in df.columns}
        X_list.append(row)
        y_list.append(int(target_value))
    return pd.DataFrame(X_list), np.array(y_list)

X_train, y_train = build_Xy(train_scaled, FEATURE_SIGNALS, TARGET_SIGNAL)
X_test,  y_test  = build_Xy(test_scaled,  FEATURE_SIGNALS, TARGET_SIGNAL)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"y_train balance: {np.bincount(y_train)}")
print(f"y_test  balance: {np.bincount(y_test)}")

X_train: (3298, 8) | X_test: (1009, 8)
y_train balance: [2163 1135]
y_test  balance: [724 285]


In [14]:
X_train.head(5)


,X,Y,Z,enmo,anglez,non-wear_flag,light,weekday
0,0 -0.362916 1 -0.195976 2 -0.16357...,0 -0.147957 1 0.517750 2 -0.41849...,0 0.054546 1 -0.338792 2 -0.40336...,0 0.421326 1 0.655773 2 1.81464...,0 0.049123 1 -0.141571 2 -0.16635...,0 0.0 1 0.0 2 0.0 3 0.0 4 ...,0 0.549546 1 0.526281 2 -0.06727...,0 4.0 1 4.0 2 4.0 3 4.0 4 ...
1,0 1.062204 1 1.080659 2 -0.15136...,0 -0.834759 1 -0.794093 2 -0.56004...,0 0.736609 1 0.439905 2 1.05116...,0 -0.256505 1 -0.225628 2 -0.43175...,0 0.866481 1 0.707613 2 0.97513...,0 0.0 1 0.0 2 0.0 3 0.0 4 ...,0 -0.532352 1 -0.528763 2 -0.52691...,0 1.0 1 1.0 2 1.0 3 1.0 4 ...
2,0 1.016709 1 -0.397746 2 -0.53783...,0 -0.626197 1 -0.385424 2 -0.47791...,0 1.215976 1 -0.197581 2 0.18493...,0 -0.088625 1 -0.472020 2 -0.58418...,0 1.005960 1 -0.078117 2 0.23276...,0 0.0 1 0.0 2 0.0 3 0.0 4 ...,0 1.459738 1 1.452811 2 1.44175...,0 3.0 1 3.0 2 3.0 3 3.0 4 ...
3,0 0.466793 1 0.437749 2 0.59050...,0 0.189711 1 0.034730 2 0.35276...,0 2.450671 1 2.558863 2 0.58158...,0 -0.169150 1 -0.348129 2 -0.26461...,0 1.164293 1 1.175778 2 0.87198...,0 0.0 1 0.0 2 0.0 3 0.0 4 ...,0 -0.659260 1 -0.656805 2 -0.64191...,0 5.0 1 5.0 2 5.0 3 5.0 4 ...
4,0 0.934145 1 0.087142 2 0.83957...,0 0.708510 1 0.238960 2 -0.13844...,0 0.465337 1 0.284776 2 -0.77726...,0 1.772592 1 0.449817 2 -0.59821...,0 0.715052 1 0.264084 2 -0.24228...,0 0.0 1 0.0 2 0.0 3 0.0 4 ...,0 0.450971 1 0.450971 2 0.45097...,0 5.0 1 5.0 2 5.0 3 5.0 4 ...


In [15]:
X_train.to_csv('../1.DATASET/TS_CLSS.csv', index=False)

In [ ]:
y_train.shape

(3298,)

In [16]:
# ── VERIFY BEFORE FITTING ─────────────────────────────────────────────────────

# 1. Shape
print("X_train shape :", X_train.shape)
print("X_test shape  :", X_test.shape)

# 2. Cell type — must be pd.Series
print("Cell type     :", type(X_train.iloc[0, 0]))

# 3. All time series same length — KNN requires this
lengths = [[len(X_train.iloc[i, j]) for j in range(X_train.shape[1])]
           for i in range(min(10, len(X_train)))]
flat    = [l for row in lengths for l in row]
print("Unique lengths:", set(flat))          # must be exactly ONE value

# 4. No NaN inside any series
has_nan = any(X_train.iloc[i, j].isna().any()
              for i in range(len(X_train))
              for j in range(X_train.shape[1]))
print("Any NaN in X  :", has_nan)            # must be False

# 5. Class balance
print("y_train balance:", np.bincount(y_train))
print("y_test  balance:", np.bincount(y_test))

X_train shape : (3298, 8)
X_test shape  : (1009, 8)
Cell type     : <class 'pandas.core.series.Series'>
Unique lengths: {200}
Any NaN in X  : False
y_train balance: [2163 1135]
y_test  balance: [724 285]


# KNN

## euclidean

In [17]:
from sklearn.model_selection import RandomizedSearchCV, KFold

random_state = 42

param_list = {
    'n_neighbors': [4],
    'weights'    : ['distance'],
    'distance'   : ['euclidean'],
    'n_jobs'     : [-1]
}

random_search = RandomizedSearchCV(
    KNeighborsTimeSeriesClassifier(),
    param_distributions = param_list,
    cv                  = KFold(n_splits=5, shuffle=True, random_state=random_state),
    scoring             = 'f1_macro',   # correct for imbalanced
    n_jobs              = -1,
    refit               = True,
    n_iter              = 10,
    random_state        = random_state
)

random_search.fit(X_train, y_train)



KeyboardInterrupt: 

In [ ]:
# ── EVALUATE ──────────────────────────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix

y_pred = random_search.predict(X_test)

print("Best params:", random_search.best_params_)
print("Best CV f1_macro:", round(random_search.best_score_, 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.metrics import (
    f1_score, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, classification_report
)
import matplotlib.pyplot as plt

def print_classification_results(best_model, X_test, y_test, model_name="Model"):
    """
    Print full classification results:
    - F1 score
    - Confusion matrix plot
    - ROC curve plot
    - Classification report
    - Confusion matrix breakdown
    """

    # ── Predictions ───────────────────────────────────────────────────────
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    test_f1 = f1_score(y_test, y_pred, average="weighted")
    print(f"{'='*60}")
    print(f"  Model       : {model_name}")
    print(f"  Test F1     : {test_f1:.4f}")
    print(f"{'='*60}")

    # ── Confusion matrix ──────────────────────────────────────────────────
    cm   = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f"Results — {model_name}", fontsize=13, fontweight="bold")

    disp.plot(cmap="Blues", ax=axes[0])
    axes[0].set_title("Confusion Matrix")
    axes[0].grid(False)

    # ── ROC curve ─────────────────────────────────────────────────────────
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc     = auc(fpr, tpr)

    axes[1].plot(fpr, tpr, color="#7F77DD", linewidth=2,
                 label=f"AUC = {roc_auc:.3f}")
    axes[1].plot([0, 1], [0, 1], "k--", linewidth=0.8, label="Random baseline")
    axes[1].set_xlabel("False Positive Rate")
    axes[1].set_ylabel("True Positive Rate")
    axes[1].set_title(f"ROC Curve")
    axes[1].legend(loc="lower right")
    axes[1].spines["top"].set_visible(False)
    axes[1].spines["right"].set_visible(False)

    plt.tight_layout()
    plt.show()

    # ── Classification report ─────────────────────────────────────────────
    print(classification_report(
        y_test, y_pred,
        target_names=["Class 0", "Class 1"]
    ))

    # ── Confusion matrix breakdown ────────────────────────────────────────
    print(f"  Confusion Matrix breakdown:")
    print(f"    TN (correct negatives) : {cm[0, 0]}")
    print(f"    FP (false positives)   : {cm[0, 1]}")
    print(f"    FN (false negatives)   : {cm[1, 0]}")
    print(f"    TP (correct positives) : {cm[1, 1]}")
    print(f"    AUC                    : {roc_auc:.4f}")
    print(f"{'='*60}")

In [ ]:
print_classification_results(
    best_model  = random_search.best_estimator_,
    X_test      = X_test,
    y_test      = y_test,
    model_name  = "KNN EUCLIDEAN"
)

## DTW 

In [ ]:
random_state = 42

param_list = {
    "n_neighbors" : [4],
    "weights"     : ["distance"],
    "distance"    : ["dtw"],
    "distance_params": [
        {"window": 0.05},   # very tight band — 10 steps
        {"window": 0.10},   # standard recommendation — 20 steps
        {"window": 0.20},   # looser — 40 steps
    ],
    "n_jobs": [-1]
}

random_search_dtw = RandomizedSearchCV(
    KNeighborsTimeSeriesClassifier(),
    param_distributions = param_list,
    cv                  = KFold(n_splits=5, shuffle=True, random_state=random_state),
    scoring             = "f1_macro",
    n_jobs              = -1,
    refit               = True,
    n_iter              = 3,            # one per window size
    random_state        = random_state
)

random_search_dtw.fit(X_train, y_train)

In [ ]:
# ── EVALUATE ──────────────────────────────────────────────────────────────────


y_pred_dtw = random_search_dtw.predict(X_test)

print("Best params  :", random_search_dtw.best_params_)
print("Best CV f1   :", round(random_search_dtw.best_score_, 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_dtw))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_dtw))

In [ ]:
print_classification_results(
    best_model  = random_search_dtw.best_estimator_,
    X_test      = X_test,
    y_test      = y_test,
    model_name  = "KNN DTM"
)